In [ ]:
# Cell 1: Install Python dependencies and setup cloudflared binary
!pip install -q sentence-transformers rank-bm25 scipy numpy fastapi uvicorn pydantic torch nest-asyncio
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
# ==============================================================================
# Cell 2: Colab GPU Reranker Microservice (Async & Thread-Safe)
# ==============================================================================
import re
import math
import logging
import numpy as np
import torch
import subprocess
import threading
import time
import uuid
from scipy.special import expit
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("colab_reranker")

TOP_VENUES = [v.upper() for v in [
    "ICSE", "FSE", "ESEC", "ASE", "ISSTA", "PLDI", "POPL", "OOPSLA",
    "SOSP", "OSDI", "USENIX", "NSDI", "SIGCOMM", "MOBICOM", "MOBISYS", "INFOCOM",
    "SIGMOD", "VLDB", "ICDE", "PODS", "NEURIPS", "NIPS", "ICML", "ICLR", "AAAI", "IJCAI",
    "CVPR", "ICCV", "ECCV", "ACL", "EMNLP", "NAACL", "COLING",
    "KDD", "WWW", "THE WEB CONFERENCE", "WSDM", "SIGIR", "CIKM",
    "CCS", "IEEE S&P", "OAKLAND", "NDSS", "CRYPTO", "EUROCRYPT", "ASIACRYPT",
    "CHI", "UIST", "CSCW", "UBICOMP", "DAC", "DATE", "MICRO", "ISCA", "HPCA",
    "SC", "IPDPS", "PPOPP", "ICRA", "IROS", "RSS", "SIGGRAPH",
    "TSE", "TOSEM", "TOPLAS", "JACM", "CACM", "TPAMI", "TNNLS", "TKDE", "TOCS", "JMLR", "TDSC", "TNSM",
    "TRANSACTIONS ON SOFTWARE ENGINEERING",
    "TRANSACTIONS ON SOFTWARE ENGINEERING AND METHODOLOGY",
    "TRANSACTIONS ON PATTERN ANALYSIS AND MACHINE INTELLIGENCE",
    "TRANSACTIONS ON NEURAL NETWORKS AND LEARNING SYSTEMS",
    "TRANSACTIONS ON KNOWLEDGE AND DATA ENGINEERING",
    "TRANSACTIONS ON COMPUTER SYSTEMS",
    "JOURNAL OF THE ACM", "COMMUNICATIONS OF THE ACM",
    "VLDB JOURNAL", "ARTIFICIAL INTELLIGENCE", "MACHINE LEARNING",
    "JOURNAL OF MACHINE LEARNING RESEARCH", "ACM COMPUTING SURVEYS",
    "TRANSACTIONS ON DEPENDABLE AND SECURE COMPUTING",
    "TRANSACTIONS ON NETWORK AND SERVICE MANAGEMENT",
    "ANNALS OF MATHEMATICS", "INVENTIONES MATHEMATICAE", "ACTA MATHEMATICA",
    "JOURNAL OF THE AMERICAN MATHEMATICAL SOCIETY", "DUKE MATHEMATICAL JOURNAL",
    "ADVANCES IN MATHEMATICS", "COMMUNICATIONS ON PURE AND APPLIED MATHEMATICS", "MATHEMATISCHE ANNALEN",
    "ANNALS OF STATISTICS", "JOURNAL OF THE ROYAL STATISTICAL SOCIETY",
    "JOURNAL OF THE AMERICAN STATISTICAL ASSOCIATION", "BIOMETRIKA", "STATISTICAL SCIENCE",
    "PHYSICAL REVIEW LETTERS", "PHYSICAL REVIEW X", "NATURE PHYSICS", "REVIEWS OF MODERN PHYSICS",
    "JOURNAL OF HIGH ENERGY PHYSICS", "APPLIED PHYSICS LETTERS",
    "PHYSICAL REVIEW A", "PHYSICAL REVIEW B", "PHYSICAL REVIEW C", "PHYSICAL REVIEW D", "PHYSICAL REVIEW E",
    "JOURNAL OF THE AMERICAN CHEMICAL SOCIETY", "ANGEWANDTE CHEMIE", "CHEMICAL REVIEWS",
    "NATURE CHEMISTRY", "ACS NANO", "NANO LETTERS", "CHEM", "ADVANCED MATERIALS",
    "CELL", "NATURE BIOTECHNOLOGY", "NATURE GENETICS", "NATURE CELL BIOLOGY", "MOLECULAR CELL",
    "GENOME RESEARCH", "GENES & DEVELOPMENT", "EMBO JOURNAL", "NEW ENGLAND JOURNAL OF MEDICINE",
    "THE LANCET", "JAMA", "BMJ", "NATURE MEDICINE", "THE LANCET ONCOLOGY", "ANNALS OF INTERNAL MEDICINE", "PLOS MEDICINE",
    "IEEE TRANSACTIONS ON INDUSTRIAL ELECTRONICS", "IEEE TRANSACTIONS ON POWER SYSTEMS",
    "IEEE TRANSACTIONS ON SMART GRID", "IEEE TRANSACTIONS ON SIGNAL PROCESSING",
    "IEEE TRANSACTIONS ON COMMUNICATIONS", "IEEE TRANSACTIONS ON CONTROL SYSTEMS TECHNOLOGY",
    "AUTOMATICA", "CONTROL ENGINEERING PRACTICE", "NATURE MATERIALS", "ADVANCED FUNCTIONAL MATERIALS",
    "ACTA MATERIALIA", "MATERIALS TODAY", "PROGRESS IN MATERIALS SCIENCE",
    "ISSCC", "IEDM", "VLSI SYMPOSIUM", "APCCAS", "ISCAS", "RFIC SYMPOSIUM",
    "ASME JOURNAL OF MECHANICAL DESIGN", "ASME JOURNAL OF MANUFACTURING SCIENCE AND ENGINEERING",
    "INTERNATIONAL JOURNAL OF MACHINE TOOLS AND MANUFACTURE", "MECHANISM AND MACHINE THEORY",
    "JOURNAL OF STRUCTURAL ENGINEERING", "ENGINEERING STRUCTURES", "CEMENT AND CONCRETE RESEARCH", "AUTOMATION IN CONSTRUCTION",
    "AMERICAN ECONOMIC REVIEW", "ECONOMETRICA", "QUARTERLY JOURNAL OF ECONOMICS",
    "JOURNAL OF POLITICAL ECONOMY", "REVIEW OF ECONOMIC STUDIES", "JOURNAL OF FINANCE",
    "JOURNAL OF FINANCIAL ECONOMICS", "REVIEW OF FINANCIAL STUDIES", "ACADEMY OF MANAGEMENT JOURNAL",
    "ACADEMY OF MANAGEMENT REVIEW", "STRATEGIC MANAGEMENT JOURNAL", "MANAGEMENT SCIENCE", "ORGANIZATION SCIENCE",
    "PSYCHOLOGICAL SCIENCE", "ANNUAL REVIEW OF PSYCHOLOGY", "JOURNAL OF PERSONALITY AND SOCIAL PSYCHOLOGY",
    "COGNITION", "NATURE CLIMATE CHANGE", "GLOBAL CHANGE BIOLOGY", "ENVIRONMENTAL SCIENCE & TECHNOLOGY", "JOURNAL OF CLEANER PRODUCTION",
    "NATURE GEOSCIENCE", "GEOPHYSICAL RESEARCH LETTERS", "EARTH AND PLANETARY SCIENCE LETTERS",
    "THE ASTROPHYSICAL JOURNAL", "ASTRONOMY & ASTROPHYSICS",
    "NATURE", "SCIENCE", "PNAS", "NATURE COMMUNICATIONS", "SCIENTIFIC REPORTS"
]]

class AcademicRetrievalPipeline:
    def __init__(
        self,
        bi_encoder_name: str = "Alibaba-NLP/gte-modernbert-base",
        cross_encoder_name: str = "BAAI/bge-reranker-v2-m3",
    ):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        logger.info("Initializing models on device: %s", device)

        self.bi_encoder = SentenceTransformer(bi_encoder_name, device=device)
        self.cross_encoder = CrossEncoder(cross_encoder_name, max_length=512, device=device)
        self.corpus_papers = []
        self.bm25 = None
        self.corpus_embeddings = None

    @staticmethod
    def _camel_case_tokenize(text: str) -> list[str]:
        text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)
        return re.findall(r"\b\w+\b", text.lower())

    def index_corpus(self, papers: list[dict], batch_size: int = 32) -> None:
        self.corpus_papers = papers
        logger.info("Indexing %d papers on GPU...", len(papers))

        tokenized_corpus = [
            self._camel_case_tokenize(f"{p.get('title', '')} {p.get('abstract', '')}")
            for p in self.corpus_papers
        ]
        self.bm25 = BM25Okapi(tokenized_corpus)

        corpus_texts = [
            f"Title: {p.get('title', '')} | Abstract: {p.get('abstract', '')}"
            for p in self.corpus_papers
        ]
        self.corpus_embeddings = self.bi_encoder.encode(
            corpus_texts,
            batch_size=batch_size,
            show_progress_bar=False,
            normalize_embeddings=True,
        )

    def _stage1_hybrid_search(self, dynamic_config: dict, rrf_k: int = 60) -> list[dict]:
        n_papers = len(self.corpus_papers)
        rrf_scores = np.zeros(n_papers)
        all_aspect_queries = dynamic_config.get("DYNAMIC_ASPECTS", []) + [dynamic_config.get("CLEAN_INTENT_QUERY", "")]

        for q_str in all_aspect_queries:
            q_tokens = self._camel_case_tokenize(q_str)
            bm25_doc_scores = self.bm25.get_scores(q_tokens)
            bm25_ranks = np.argsort(-bm25_doc_scores)

            q_emb = self.bi_encoder.encode([q_str], normalize_embeddings=True)
            dense_doc_scores = np.dot(self.corpus_embeddings, q_emb.T).squeeze()
            dense_ranks = np.argsort(-dense_doc_scores)

            for rank_idx, doc_idx in enumerate(bm25_ranks):
                rrf_scores[doc_idx] += 1.0 / (rrf_k + rank_idx + 1)
            for rank_idx, doc_idx in enumerate(dense_ranks):
                rrf_scores[doc_idx] += 1.0 / (rrf_k + rank_idx + 1)

        stage1_final_scores = np.copy(rrf_scores)
        baseline_models = [m.lower() for m in dynamic_config.get("BASELINE_MODELS", [])]
        penalty_terms = [p.lower() for p in dynamic_config.get("PENALTY_TERMS", [])]

        for i, p in enumerate(self.corpus_papers):
            t_clean = p.get("title", "").lower()
            a_clean = p.get("abstract", "").lower()
            v_clean = str(p.get("venue", "")).lower()

            if any(m in t_clean for m in baseline_models if len(m) > 2):
                stage1_final_scores[i] += 0.030

            c_count = p.get("citations", 0)
            if c_count > 0:
                stage1_final_scores[i] += min(0.020, math.log10(c_count + 1) * 0.005)

            if any(pt in a_clean or pt in t_clean for pt in penalty_terms):
                stage1_final_scores[i] -= 0.025

            if any(v in v_clean for v in TOP_VENUES):
                stage1_final_scores[i] += 0.015

            try:
                year = int(p.get("year", 2026))
            except (ValueError, TypeError):
                year = 2026

            age = max(0, 2026 - year)
            stage1_final_scores[i] -= min(0.020, age * 0.002)

        scored_papers = []
        for i, p in enumerate(self.corpus_papers):
            paper_copy = dict(p)
            paper_copy["stage1_score"] = float(stage1_final_scores[i])
            scored_papers.append(paper_copy)

        return sorted(scored_papers, key=lambda x: x["stage1_score"], reverse=True)

    def _stage2_additive_rerank(self, top_candidates: list[dict], dynamic_config: dict, alpha: float = 0.70) -> list[dict]:
        all_aspect_queries = dynamic_config.get("DYNAMIC_ASPECTS", []) + [dynamic_config.get("CLEAN_INTENT_QUERY", "")]
        ce_max_p_scores = []

        for paper in top_candidates:
            doc_text = f"Title: {paper.get('title', '')} | Abstract: {paper.get('abstract', '')}"
            pairs = [[aspect_q, doc_text] for aspect_q in all_aspect_queries]

            raw_logits = self.cross_encoder.predict(pairs, show_progress_bar=False)
            probs = expit(raw_logits)
            max_score = float(np.max(probs))
            ce_max_p_scores.append(max_score)
            paper["ce_score"] = max_score

        s1_raw = np.array([p["stage1_score"] for p in top_candidates])
        s1_min, s1_max = s1_raw.min(), s1_raw.max()
        s1_norm = (s1_raw - s1_min) / (s1_max - s1_min + 1e-8)

        ce_norm = np.array(ce_max_p_scores)
        blended_scores = (alpha * ce_norm) + ((1.0 - alpha) * s1_norm)

        for idx, p in enumerate(top_candidates):
            p["final_blended_score"] = float(blended_scores[idx])

        return sorted(top_candidates, key=lambda x: x["final_blended_score"], reverse=True)

    def retrieve(self, dynamic_config: dict, top_k_stage1: int = 600, alpha: float = 0.70) -> list[dict]:
        if not self.corpus_papers:
            return []
        stage1_ranked = self._stage1_hybrid_search(dynamic_config)
        top_candidates = stage1_ranked[:top_k_stage1]
        remainder = stage1_ranked[top_k_stage1:]
        stage2_ranked = self._stage2_additive_rerank(top_candidates, dynamic_config, alpha=alpha)
        return stage2_ranked + remainder

# Load pipeline onto GPU
pipeline = AcademicRetrievalPipeline()

app = FastAPI()

class RerankRequest(BaseModel):
    dict_corpus: list[dict]
    dynamic_config: dict

# In-memory store for async jobs
job_store = {}

@app.post("/rerank_async")
def rerank_async_endpoint(payload: RerankRequest):
    job_id = str(uuid.uuid4())
    job_store[job_id] = {"status": "processing", "result": []}

    def background_task():
        try:
            pipeline.index_corpus(payload.dict_corpus)
            ranked = pipeline.retrieve(payload.dynamic_config)
            job_store[job_id] = {"status": "completed", "result": ranked}
            logger.info(f"Job {job_id} completed successfully.")
        except Exception as e:
            logger.error(f"Job {job_id} failed: {e}")
            job_store[job_id] = {"status": "failed", "error": str(e)}

    # Start the heavy processing in a background thread so the API can respond immediately
    threading.Thread(target=background_task, daemon=True).start()

    return {"job_id": job_id, "status": "processing"}

@app.get("/rerank_status/{job_id}")
def rerank_status(job_id: str):
    if job_id not in job_store:
        return {"status": "not_found"}
    return job_store[job_id]

# ==============================================================================
# Start Server & Tunnel Safely in Colab (Threaded to prevent asyncio crashes)
# ==============================================================================
def start_cloudflared():
    time.sleep(3)  # Give FastAPI a moment to spin up
    command = ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8001"]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    for line in process.stdout:
        url_match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if url_match:
            print(f"\n=======================================================")
            print(f"🚀 COPY THIS BASE URL INTO YOUR LOCAL .env FILE:")
            print(f"COLAB_RERANKER_URL={url_match.group(0)}")
            print(f"=======================================================\n")
            break  # URL found, stop parsing logs

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8001)

threading.Thread(target=run_server, daemon=True).start()
threading.Thread(target=start_cloudflared, daemon=True).start()

try:
    print("Initializing async server and tunnel... please wait.")
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nServer stopped by user.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/14.0k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  298MB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

INFO:     Started server process [554]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


Initializing async server and tunnel... please wait.

🚀 COPY THIS BASE URL INTO YOUR LOCAL .env FILE:
COLAB_RERANKER_URL=https://crimes-chips-gibson-fireplace.trycloudflare.com

INFO:     117.200.53.211:0 - "POST /rerank_async HTTP/1.1" 200 OK
INFO:     117.200.53.211:0 - "GET /rerank_status/7950f95e-66a7-4d19-adb2-8fc0a847741b HTTP/1.1" 200 OK
INFO:     117.200.53.211:0 - "GET /rerank_status/7950f95e-66a7-4d19-adb2-8fc0a847741b HTTP/1.1" 200 OK
INFO:     117.200.53.211:0 - "GET /rerank_status/7950f95e-66a7-4d19-adb2-8fc0a847741b HTTP/1.1" 200 OK
INFO:     117.200.53.211:0 - "GET /rerank_status/7950f95e-66a7-4d19-adb2-8fc0a847741b HTTP/1.1" 200 OK
INFO:     117.200.53.211:0 - "GET /rerank_status/7950f95e-66a7-4d19-adb2-8fc0a847741b HTTP/1.1" 200 OK
INFO:     117.200.53.211:0 - "GET /rerank_status/7950f95e-66a7-4d19-adb2-8fc0a847741b HTTP/1.1" 200 OK
INFO:     117.200.53.211:0 - "GET /rerank_status/7950f95e-66a7-4d19-adb2-8fc0a847741b HTTP/1.1" 200 OK
INFO:     117.200.53.211:0 - "GET /